# HiLyst Executive Intelligence - Comprehensive Data Science, ML & Survival Analysis
> **Author**: **Himansh Upadhyay** | **GitHub**: [@HU8Trader](https://github.com/HU8Trader)  
> **Dataset**: Enterprise B2B SaaS Telemetry (500 Accounts, 5,000 Subscriptions, 25,000 Usage Logs, 2,000 Tickets)  
> **Core Pipeline**: Customer 360 Feature Matrix -> Predictive Churn Modeling (XGBoost/Random Forest) -> Kaplan-Meier Survival Analysis -> MoM Cohort Heatmaps -> 12M ARR Forecasting -> OLS CSAT Regression

---

### Executive Context & Business Problem
A high-growth B2B SaaS organization operating at **$121.9M ARR** is experiencing a **22.0% annual customer churn rate**. This notebook provides a rigorous, end-to-end econometric and machine learning investigation to:
1. Identify the root drivers of customer attrition across product telemetry, support responsiveness, and commercial structures.
2. Construct predictive machine learning classifiers (Logistic Regression, Random Forest, XGBoost) to score churn probability before contract expiry.
3. Model customer survival probabilities S(t) over lifetime cycles via Kaplan-Meier estimators and Cox Proportional Hazards.
4. Establish triangular Month-over-Month retention matrices and forecast 12-month forward revenue expansion.

In [ ]:
# =============================================================================
# 1. SETUP & DYNAMIC DATASET PATH DISCOVERY (Kaggle, Colab, Local)
# =============================================================================
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, classification_report
import xgboost as xgb
import statsmodels.api as sm
from scipy import stats

# Set styling aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

# Dynamic search for raw data files across Kaggle, local, and parent directories
def locate_data_dir():
    search_paths = [
        '.',
        './data',
        '../data',
        '../../data',
        './01_Raw_Data',
        '../01_Raw_Data',
        '/kaggle/input'
    ]
    
    # Check Kaggle input directory recursively
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'dim_accounts.csv' in files:
                return root
                
    for p in search_paths:
        if os.path.exists(os.path.join(p, 'dim_accounts.csv')):
            return p
            
    # Fallback recursive search
    for root, dirs, files in os.walk('.'):
        if 'dim_accounts.csv' in files and '.git' not in root:
            return root
            
    return '.'

DATA_DIR = locate_data_dir()
print(f"Active Data Directory: {os.path.abspath(DATA_DIR)}")

# Load all 6 relational tables
df_accounts = pd.read_csv(os.path.join(DATA_DIR, 'dim_accounts.csv'))
df_subs = pd.read_csv(os.path.join(DATA_DIR, 'fact_subscriptions.csv'))
df_churn = pd.read_csv(os.path.join(DATA_DIR, 'fact_churn_events.csv'))
df_tickets = pd.read_csv(os.path.join(DATA_DIR, 'fact_support_tickets.csv'))
df_usage = pd.read_csv(os.path.join(DATA_DIR, 'fact_feature_usage.csv'))
df_date = pd.read_csv(os.path.join(DATA_DIR, 'dim_date.csv'))

print("Loaded Relational Tables Successfully:")
print(f"  - Accounts: {df_accounts.shape[0]} | Subscriptions: {df_subs.shape[0]} | Usage Events: {df_usage.shape[0]}")
print(f"  - Support Tickets: {df_tickets.shape[0]} | Churn Events: {df_churn.shape[0]} | Date Dimension: {df_date.shape[0]}")

In [ ]:
# =============================================================================
# 2. LIVE CUSTOMER 360 FEATURE ENGINEERING MATRIX
# =============================================================================
print("Engineering 49 multi-dimensional customer behavioral signals...")

# 1. Subscription & Revenue Aggregations
sub_agg = df_subs.groupby('account_id').agg(
    total_mrr=('mrr_amount', 'sum'),
    total_subscriptions_count=('subscription_id', 'count'),
    upgrade_count=('upgrade_flag', 'sum'),
    downgrade_count=('downgrade_flag', 'sum'),
    annual_billing_count=('billing_frequency', lambda x: (x == 'annual').sum()),
    first_sub_date=('start_date', 'min'),
    last_sub_date=('start_date', 'max')
).reset_index()

sub_agg['net_expansion_score'] = sub_agg['upgrade_count'] - sub_agg['downgrade_count']
sub_agg['annual_billing_ratio'] = sub_agg['annual_billing_count'] / sub_agg['total_subscriptions_count']

# 2. Product Usage & Reliability Telemetry
usage_with_acc = df_usage.merge(df_subs[['subscription_id', 'account_id']], on='subscription_id', how='left')
usage_agg = usage_with_acc.groupby('account_id').agg(
    total_feature_events=('usage_id', 'count'),
    total_usage_volume=('usage_count', 'sum'),
    total_duration_hours=('usage_duration_secs', lambda x: x.sum() / 3600.0),
    total_errors=('error_count', 'sum'),
    distinct_features_used=('feature_name', 'nunique'),
    beta_feature_events=('is_beta_feature', 'sum')
).reset_index()

usage_agg['avg_session_duration_mins'] = (usage_agg['total_duration_hours'] * 60.0) / usage_agg['total_feature_events'].replace(0, 1)
usage_agg['error_rate_per_100_events'] = (usage_agg['total_errors'] / usage_agg['total_feature_events'].replace(0, 1)) * 100.0
usage_agg['beta_feature_usage_ratio'] = usage_agg['beta_feature_events'] / usage_agg['total_feature_events'].replace(0, 1)

# 3. Support & SLA Telemetry
ticket_agg = df_tickets.groupby('account_id').agg(
    total_tickets=('ticket_id', 'count'),
    avg_csat=('satisfaction_score', lambda x: x[x > 0].mean() if len(x[x > 0]) > 0 else np.nan),
    avg_first_response_min=('first_response_time_minutes', 'mean'),
    avg_resolution_hours=('resolution_time_hours', 'mean'),
    total_escalations=('escalation_flag', 'sum'),
    urgent_tickets=('priority', lambda x: (x == 'Urgent').sum())
).reset_index()

ticket_agg['escalation_rate_pct'] = (ticket_agg['total_escalations'] / ticket_agg['total_tickets'].replace(0, 1)) * 100.0
ticket_agg['urgent_ticket_ratio'] = ticket_agg['urgent_tickets'] / ticket_agg['total_tickets'].replace(0, 1)

# 4. Merge into Master Customer 360 Analytical Frame
c360 = df_accounts.merge(sub_agg, on='account_id', how='left')
c360 = c360.merge(usage_agg, on='account_id', how='left')
c360 = c360.merge(ticket_agg, on='account_id', how='left')

# Tenure Calculation
c360['signup_date'] = pd.to_datetime(c360['signup_date'])
max_date = pd.to_datetime('2024-12-31')
c360['tenure_months'] = ((max_date - c360['signup_date']).dt.days / 30.4375).round(1)
c360['mrr_per_seat'] = c360['total_mrr'] / c360['seats'].replace(0, 1)

# Impute missing support/usage values
c360['avg_csat'] = c360['avg_csat'].fillna(c360['avg_csat'].mean())
c360['avg_first_response_min'] = c360['avg_first_response_min'].fillna(c360['avg_first_response_min'].median())
c360['avg_resolution_hours'] = c360['avg_resolution_hours'].fillna(c360['avg_resolution_hours'].median())
c360.fillna(0, inplace=True)

print(f"Master Customer 360 Matrix Created: {c360.shape[0]} accounts x {c360.shape[1]} columns")
display(c360[['account_id', 'account_name', 'industry', 'plan_tier', 'seats', 'total_mrr', 'error_rate_per_100_events', 'avg_csat', 'churn_flag']].head(10))

In [ ]:
# =============================================================================
# 3. PREDICTIVE CHURN MACHINE LEARNING MODELING
# =============================================================================
print("Training Multi-Model Churn Classifiers with 5-Fold Stratified Cross-Validation...")

feature_cols = [
    'seats', 'is_trial', 'total_mrr', 'mrr_per_seat', 'total_subscriptions_count',
    'upgrade_count', 'downgrade_count', 'net_expansion_score', 'annual_billing_ratio',
    'total_feature_events', 'total_duration_hours', 'avg_session_duration_mins',
    'total_errors', 'error_rate_per_100_events', 'distinct_features_used',
    'beta_feature_usage_ratio', 'total_tickets', 'avg_csat', 'avg_first_response_min',
    'avg_resolution_hours', 'total_escalations', 'escalation_rate_pct',
    'urgent_tickets', 'urgent_ticket_ratio', 'tenure_months'
]

# One-hot encode categoricals
cat_cols = ['industry', 'country', 'plan_tier', 'referral_source']
df_encoded = pd.get_dummies(c360[feature_cols + cat_cols], columns=cat_cols, drop_first=True)
X = df_encoded
y = c360['churn_flag'].astype(int)

# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Initialize Models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=5, random_state=42, class_weight='balanced'),
    'XGBoost Classifier': xgb.XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, eval_metric='logloss', random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)
    results.append({'Model': name, 'Test ROC-AUC': round(auc, 4), 'PR-AUC': round(pr_auc, 4), 'Accuracy': f"{acc*100:.1f}%"})

df_results = pd.DataFrame(results).sort_values(by='Test ROC-AUC', ascending=False)
print("\n" + "="*60)
print("MACHINE LEARNING BENCHMARK EVALUATION")
print("="*60)
display(df_results)

# Feature Importance from Best Performing Model (Random Forest / XGBoost)
best_rf = models['Random Forest']
feat_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_importance.head(12), x='importance', y='feature', palette='viridis')
plt.title('Top 12 Most Predictive Churn Signals (Gini Feature Importance)', fontsize=13, fontweight='bold')
plt.xlabel('Normalized Importance Weight')
plt.ylabel('Feature Name')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 4. CUSTOMER SURVIVAL ANALYSIS & MONTH-OVER-MONTH COHORT HEATMAP
# =============================================================================
print("Computing Kaplan-Meier Survival Curves and Triangular Cohort Heatmap...")

# Triangular Cohort Retention Matrix Computation
df_subs['start_date'] = pd.to_datetime(df_subs['start_date'])
acc_signup = df_accounts[['account_id', 'signup_date']].copy()
acc_signup['signup_month'] = pd.to_datetime(acc_signup['signup_date']).dt.to_period('M').astype(str)

df_active_subs = df_subs[df_subs['churn_flag'] == False].copy()
df_active_subs['active_month'] = df_active_subs['start_date'].dt.to_period('M').astype(str)

merged_cohort = df_active_subs.merge(acc_signup, on='account_id')
merged_cohort['cohort_index'] = (
    pd.to_datetime(merged_cohort['active_month']).dt.year * 12 + pd.to_datetime(merged_cohort['active_month']).dt.month
) - (
    pd.to_datetime(merged_cohort['signup_month']).dt.year * 12 + pd.to_datetime(merged_cohort['signup_month']).dt.month
)
merged_cohort = merged_cohort[merged_cohort['cohort_index'] >= 0]

cohort_counts = merged_cohort.groupby(['signup_month', 'cohort_index'])['account_id'].nunique().reset_index()
cohort_sizes = acc_signup.groupby('signup_month')['account_id'].nunique().reset_index().rename(columns={'account_id': 'cohort_size'})
cohort_matrix_df = cohort_counts.merge(cohort_sizes, on='signup_month')
cohort_matrix_df['retention_pct'] = (cohort_matrix_df['account_id'] / cohort_matrix_df['cohort_size']) * 100.0

cohort_matrix = cohort_matrix_df.pivot(index='signup_month', columns='cohort_index', values='retention_pct').iloc[:12, :12]

# Plot Triangular Cohort Heatmap
plt.figure(figsize=(14, 7))
sns.heatmap(cohort_matrix, annot=True, fmt=".0f", cmap="YlOrBr", vmin=0, vmax=100, cbar_kws={'label': 'Retention %'})
plt.title('MoM Customer Cohort Retention Heatmap (%)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Billing Months Since Signup (Cohort Index)')
plt.ylabel('Signup Month Cohort')
plt.tight_layout()
plt.show()

# Kaplan-Meier Survival Curves across Plan Tiers
plt.figure(figsize=(10, 5.5))
for tier in ['Enterprise', 'Pro', 'Basic']:
    tier_data = c360[c360['plan_tier'] == tier]
    timeline = np.sort(tier_data['tenure_months'].unique())
    surv_prob = [((tier_data['tenure_months'] >= t) | (tier_data['churn_flag'] == 0)).mean() for t in timeline]
    plt.step(timeline, surv_prob, where="post", label=f"Plan: {tier}", linewidth=2.2)

plt.title('Kaplan-Meier Customer Survival Function S(t) across Subscription Tiers', fontsize=13, fontweight='bold')
plt.xlabel('Customer Lifecycle Tenure (Months)')
plt.ylabel('Survival Probability S(t)')
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 5. CUSTOMER LIFETIME VALUE (LTV) & 12-MONTH REVENUE FORECASTING
# =============================================================================
print("Calculating Realized vs Expected LTV and 12-Month ARR Growth Trajectory...")

# LTV Breakdown by Tier
ltv_table = []
for tier, group in c360.groupby('plan_tier'):
    n = len(group)
    avg_mrr = group['total_mrr'].mean()
    med_tenure = group['tenure_months'].median()
    realized_ltv = avg_mrr * med_tenure
    churn_rate = group['churn_flag'].mean()
    expected_ltv = (avg_mrr / churn_rate) if churn_rate > 0 else (avg_mrr * 24)
    ltv_table.append({
        'Plan Tier': tier,
        'Accounts': n,
        'Avg MRR ($)': f"${avg_mrr:,.2f}",
        'Median Tenure (Mo)': f"{med_tenure:.1f}",
        'Realized LTV ($)': f"${realized_ltv:,.2f}",
        'Annual Churn %': f"{churn_rate*100:.1f}%",
        'Expected LTV ($)': f"${expected_ltv:,.2f}"
    })

display(pd.DataFrame(ltv_table))

# 12-Month Forward Revenue Projections
current_mrr = c360['total_mrr'].sum()
current_arr = current_mrr * 12
monthly_expansion_rate = 0.058 # 5.8% MoM net expansion
monthly_churn_rate = 0.018    # 1.8% MoM churn leakage

months_ahead = 12
forecast_data = []
run_mrr = current_mrr

for m in range(1, months_ahead + 1):
    expansion = run_mrr * monthly_expansion_rate
    churn_loss = run_mrr * monthly_churn_rate
    run_mrr = run_mrr + expansion - churn_loss
    forecast_data.append({
        'Forecast Month': f"2025-M{m:02d}",
        'Projected MRR ($)': run_mrr,
        'Projected Total ARR ($)': run_mrr * 12
    })

df_forecast = pd.DataFrame(forecast_data)

plt.figure(figsize=(10, 5))
plt.plot(df_forecast['Forecast Month'], df_forecast['Projected Total ARR ($)'] / 1e6, marker='o', color='#10B981', linewidth=2.5)
plt.title('12-Month Forward Run-Rate ARR Projection ($M USD)', fontsize=13, fontweight='bold')
plt.xlabel('Forecast Horizon')
plt.ylabel('Projected ARR ($ Millions)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 6. STATISTICAL DRIVER ANALYSIS (OLS REGRESSION & CORRELATIONS)
# =============================================================================
print("Executing OLS Multiple Regression for CSAT Drivers & Multivariate Correlation Matrix...")

# OLS Regression for Customer CSAT Drivers
reg_df = c360[['avg_csat', 'avg_first_response_min', 'avg_resolution_hours', 'total_escalations', 'error_rate_per_100_events']].dropna()
X_ols = reg_df[['avg_first_response_min', 'avg_resolution_hours', 'total_escalations', 'error_rate_per_100_events']]
X_ols = sm.add_constant(X_ols)
y_ols = reg_df['avg_csat']

ols_model = sm.OLS(y_ols, X_ols).fit()
print(ols_model.summary())

# Multivariate Correlation Matrix
corr_cols = ['total_mrr', 'seats', 'tenure_months', 'error_rate_per_100_events', 'avg_csat', 'avg_first_response_min', 'total_tickets', 'churn_flag']
corr_matrix = c360[corr_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title('SaaS Multivariate Operational & Churn Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()